# Вариант 16. Задача 3. Динамическое программирование

## Постановка задачи

Склад пункта реализации станков имеет вместимость 15 единиц. Пополнение склада возможно только первого числа каждого месяца. Станки привозят автотранспортом (1 рейс), причем стоимость рейса складывается из постоянных затрат 30 денежных единиц (ДЕ) и затрат на доставку каждого станка 10 ДЕ. За один рейс (и, следовательно, в месяц) может быть доставлено не более 7 станков. Затраты на хранение станка в течение месяца составляют 8 ДЕ. Ожидаемый спрос на станки приведен в табл. 1. Отсутствие требуемого количества станков на складе недопустимо. В начальный момент на складе находится 2 станков.
1. определить план заказов, минимизирующий стоимость;
2. определить границы изменения стоимости хранения станка, в которых найденный план
является оптимальным;
3. определить границы изменения постоянных затрат на совершение рейса, в которых найденный план является оптимальным;
4. определить план заказов, минимизирующий стоимость, при условии, что к концу периода
на складе должно остаться 2 станков.

Таблица 1: Ожидаемый спрос на станки по месяцам
| | 1 | 2 | 3 | 4 | 5 | 6 |
|-|---|---|---|---|---|---|
| Спрос, шт. | 5 | 2 | 1 | 9 | 9 | 2 |

## Формализация задачи

- $i$ - этапы, месяцы в задаче. При этом $i$$\in$[1, 6].
- $u_i$ - управление, количество станков, которое заказываем в $i$-й месяц. При этом $u_i$$\in$[0, 7].
- $S_i$ - состояние, количество станков на складе на начало месяца. При этом $S_i\in$[0, 15] и $S_1$ = 2.
- $w_i$ - выигрыш в $i$-м месяце, затраты в $i$-м месяце. $w_i = 30 * \mathbb{1}_{u_i > 0} + 10u_i + 8(S_i + u_i - d_i)$
- $d_i$ - спрос на станки в $i$-м месяце.
- $phi_i(S_i, u_i)$ - функция перехода. $phi_i(S_i, u_i)$ = $S_i$ + $u_i$ - $d_i$.
- $W_i(S_i)$ - функция условного оптимального выигрыша. $W_i(S_i) = \min_{u_i \in \{u \mid u \in \{0, ... , 7\},\; u + S_i \ge d_i\}} \{30 * \mathbb{1}_{u_i > 0} + 10u_i + 8(S_i + u_i - d_i) + W_{i+1}(phi_i)\}$

## Результат

### 1. Оптимальный план заказов (минимизация стоимости)

Оптимальное управление ($u_i$) (количество заказываемых станков по месяцам):

| Месяц ($i$) | Управление ($u_i$) | Состояние на начало месяца ($S_i$) | Спрос ($d_i$) | Состояние на конец месяца |
|-------------|--------------------|------------------------------------|---------------|---------------------------|
| 1           | **5**              | 2                                  | 5             | 2                         |
| 2           | **0**              | 2                                  | 2             | 0                         |
| 3           | **5**              | 0                                  | 1             | 4                         |
| 4           | **7**              | 4                                  | 9             | 2                         |
| 5           | **7**              | 2                                  | 9             | 0                         |
| 6           | **2**              | 0                                  | 2             | 0                         |

**Итоговый выигрыш (минимальные суммарные затраты):** **474** ДЕ

### 2. Границы изменения стоимости хранения станка (h)

План остаётся оптимальным при изменении стоимости хранения в пределах
**$(5 \le h \le 14)$**.

### 3. Границы изменения постоянных затрат на рейс (K)

План остаётся оптимальным при изменении постоянных затрат в пределах
**$(17 \le K \le 48)$**.

### 4. План заказов с условием остатка 2 станка в конце периода

Оптимальное управление: **[5, 0, 5, 7, 7, 4]**

**Итоговый выигрыш (минимальные затраты):** **510** ДЕ

## Приложение

In [1]:
class InventoryDP:
    def __init__(self, demand, initial_s=2, max_s=15, max_u=7, h=8, K=30, c=10):
        self.demand = demand
        self.initial_s = initial_s
        self.max_s = max_s
        self.max_u = max_u
        self.h = h
        self.K = K
        self.c = c
        self.n_stages = len(demand)
        self.W_cache = [{} for _ in range(self.n_stages + 1)]

    def calculate(self, terminal_s=None):
        self.W_cache = [{} for _ in range(self.n_stages + 1)]
        for state in range(self.max_s + 1):
            if terminal_s is None:
                self.W_cache[self.n_stages][state] = (0, None)
            else:
                val = 0 if state == terminal_s else float('inf')
                self.W_cache[self.n_stages][state] = (val, None)
        for stage in reversed(range(self.n_stages)):
            d_i = self.demand[stage]
            for s_i in range(self.max_s + 1):
                best_w = float('inf')
                best_u = None
                for u_i in range(self.max_u + 1):
                    if s_i + u_i >= d_i:
                        next_s = s_i + u_i - d_i
                        if next_s <= self.max_s:
                            order_cost = (self.K + self.c * u_i) if u_i > 0 else 0
                            holding_cost = self.h * next_s
                            current_w = order_cost + holding_cost + self.W_cache[stage+1][next_s][0]
                            if current_w < best_w:
                                best_w = current_w
                                best_u = u_i
                self.W_cache[stage][s_i] = (best_w, best_u)

    def get_plan(self):
        plan = []
        curr_s = self.initial_s
        total_cost = self.W_cache[0][curr_s][0]

        if total_cost == float('inf'):
            return None, float('inf')

        for stage in range(self.n_stages):
            u = self.W_cache[stage][curr_s][1]
            plan.append(u)
            curr_s = curr_s + u - self.demand[stage]

        return plan, total_cost

demand_16 = [5, 2, 1, 9, 9, 2]

model = InventoryDP(demand_16)
model.calculate()
base_plan, base_cost = model.get_plan()
print(f'1. Оптимальный план: {base_plan}, Затраты: {base_cost}')

def find_sensitivity(param_name, limit=100):
    stable_vals = []
    for val in range(limit):
        params = {'demand': demand_16, param_name: val}
        test_model = InventoryDP(**params)
        test_model.calculate()
        if test_model.get_plan()[0] == base_plan:
            stable_vals.append(val)
    return min(stable_vals), max(stable_vals)

print(f'2. Границы h (хранение): {find_sensitivity("h")}')
print(f'3. Границы K (рейс): {find_sensitivity("K")}')

model.calculate(terminal_s=2)
plan4, cost4 = model.get_plan()
print(f'4. План при остатке 2 в конце: {plan4}, Затраты: {cost4}')

1. Оптимальный план: [5, 0, 5, 7, 7, 2], Затраты: 474
2. Границы h (хранение): (5, 14)
3. Границы K (рейс): (17, 48)
4. План при остатке 2 в конце: [5, 0, 5, 7, 7, 4], Затраты: 510
